# MvDEC imported DEKM_F View1 (trainable, joint Eq. 11) + true bottleneck + 500-update refinement floor - REUTERS seed 42

Controlled diagnostic only (`controlled_diagnostic_not_exact_paper_truth`). View1 starts from the user-owned verified DEKM_F seed-42 checkpoint and stays trainable. Only the minimum number of refinement updates differs from the plain frozen true-bottleneck arm: the assignment-change stop may not fire before update 500.

Kaggle settings: GPU on, Internet on, attach dataset `datnguyends/mvdec-v1-true-bottleneck-largest-eigen-s42-input` (used only for the DEKM_F checkpoint). Code is cloned from GitHub at a pinned commit.

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("GPUs:", gpus)
assert gpus, "Enable a Kaggle GPU before running this diagnostic."

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys

PROTOCOL = "mvdec_2025_imported_dekm_f_view1_joint_eq11_true_bottleneck_min500_release_l2_seed42_v1"
PINNED_COMMIT = "5da3967e76a7e4634497e6e646b9f0e3e2d07d83"
REPO_URL = "https://github.com/vdat-18/mvdec_fpmax.git"
REPO = Path("/kaggle/temp/mvdec_fpmax")
OUTPUT = Path("/kaggle/working/mvdec-joint-view1-true-bottleneck-min500-seed42")
EXPECTED_CHECKPOINT_MANIFEST_SHA256 = "71e3c2d423d70bbde1b7423c91c176c1d33a4fb473e7cb6febec187fff8f0607"
EXPECTED_CHECKPOINT_WEIGHT_SHA256 = "ca55a1e995697c346d99540ddd9cc8bd8e6182a7d21d0b09410d7e76b6050702"


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


candidates = sorted(
    manifest.parent
    for manifest in Path("/kaggle/input").rglob("manifest.json")
    if (manifest.parent / "final.weights.h5").is_file()
    and file_sha256(manifest) == EXPECTED_CHECKPOINT_MANIFEST_SHA256
)
assert len(candidates) == 1, candidates
CHECKPOINT_DIR = candidates[0]
CHECKPOINT_MANIFEST = CHECKPOINT_DIR / "manifest.json"
assert file_sha256(CHECKPOINT_DIR / "final.weights.h5") == EXPECTED_CHECKPOINT_WEIGHT_SHA256
print("Checkpoint:", CHECKPOINT_DIR)

for path in (REPO, OUTPUT):
    if path.exists():
        shutil.rmtree(path)
OUTPUT.mkdir(parents=True)
subprocess.run(["git", "clone", "-q", "-b", "dev", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "checkout", "-q", PINNED_COMMIT], cwd=REPO, check=True)
TRAINING_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert TRAINING_COMMIT == PINNED_COMMIT, (TRAINING_COMMIT, PINNED_COMMIT)
assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip()
print("Training commit:", TRAINING_COMMIT)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "gdown"], check=True)
sys.path.insert(0, str(REPO / "src"))
subprocess.run(
    [sys.executable, str(REPO / "scripts/fetch_dekm_release.py"), "--datasets", "REUTERS"],
    cwd=REPO,
    check=True,
)

In [ ]:
from pipeline.dekm_checkpoint import (
    DEKM_F_REUTERS_SOURCE_SHA256,
    validate_dekm_f_view1_checkpoint,
)

validated_checkpoint = validate_dekm_f_view1_checkpoint(
    CHECKPOINT_MANIFEST,
    expected_dataset="REUTERS",
    expected_seed=42,
    expected_source_sha256=DEKM_F_REUTERS_SOURCE_SHA256,
)
assert validated_checkpoint.manifest_sha256 == EXPECTED_CHECKPOINT_MANIFEST_SHA256
assert validated_checkpoint.weights_sha256 == EXPECTED_CHECKPOINT_WEIGHT_SHA256

subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q",
        "tests/test_dekm_checkpoint.py",
        "tests/test_mvdec_independent_view2.py",
        "tests/test_mvdec_contract.py",
        "tests/test_data.py",
        "tests/test_mvdec_architecture.py",
    ],
    cwd=REPO,
    env={**os.environ, "PYTHONPATH": os.pathsep.join([str(REPO), str(REPO / "src")])},
    check=True,
)

In [ ]:
command = [
    sys.executable, "-u", "src/representation_learning/MVDEC_dense.py", "REUTERS",
    "--runs", "1", "--seed", "42",
    "--protocol", PROTOCOL,
    "--view1-manifest", str(CHECKPOINT_MANIFEST),
    "--progress-interval", "100",
    "--dataset-root", "external_repos/DEKM/datasets",
    "--public-output-dir", str(OUTPUT),
]
print("Running:", " ".join(command))
subprocess.run(command, cwd=REPO, check=True, timeout=8 * 60 * 60)

In [ ]:
for module, extra in (
    ("pipeline.mvdec_audit", ["--require-all-complete"]),
    ("pipeline.mvdec_fusion_audit", ["--output-dir", str(OUTPUT / "fusion_audit")]),
):
    subprocess.run(
        [
            sys.executable, "-m", module,
            "--root", str(OUTPUT),
            "--expected-datasets", "REUTERS",
            "--expected-protocol-id", PROTOCOL,
            "--expected-training-commit", TRAINING_COMMIT,
            "--expected-seed", "42",
            *extra,
            *(["--dataset-root", "external_repos/DEKM/datasets"] if module.endswith("mvdec_audit") else []),
        ],
        cwd=REPO,
        check=True,
    )

In [ ]:
import csv

PAPER = {"view1": (0.7642, 0.5999), "view2": (0.6501, 0.5129), "fused": (0.7813, 0.6141)}
manifest_path = next(OUTPUT.rglob("manifest.json"))
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
assert manifest["status"] == "complete" and manifest["protocol_id"] == PROTOCOL
metrics = manifest["metrics"]
log_text = (manifest_path.parent / "training_log.csv").read_text(encoding="utf-8")
stop_line = next(line for line in log_text.splitlines() if "phase:training_stop" in line)
print(stop_line)
steps = int(stop_line.split("training_steps_completed:")[1].split(",")[0].split(";")[0])
assert steps >= 500, f"Refinement stopped before the 500-update floor: {steps}"

print(f"{'rep':6s} {'ACC':>8s} {'NMI':>8s} | paper ACC / NMI")
for key in ("view1", "view2", "fused"):
    acc, nmi = metrics[f"{key}_acc"], metrics[f"{key}_nmi"]
    print(f"{key:6s} {acc:8.4f} {nmi:8.4f} | {PAPER[key][0]:.4f} / {PAPER[key][1]:.4f}")

zip_path = Path(shutil.make_archive(
    "/kaggle/working/mvdec-joint-view1-true-bottleneck-min500-seed42-results",
    "zip", OUTPUT.parent, OUTPUT.name,
))
digest = file_sha256(zip_path)
Path(str(zip_path) + ".sha256").write_text(f"{digest}  {zip_path.name}\n", encoding="ascii")
print("ZIP:", zip_path)
print("SHA-256:", digest)